# [LAB 04 데이터 다루기] - 10. 더미변수

## #01. 준비작업 

### 1. 라이브러리 참조

In [2]:
from jussam import load_data

# 데이터 프레임, 더미변수 생성 함수, 데이터 병합 함수 
from pandas import DataFrame, get_dummies, merge 

# 원-핫 인코딩 클래스 (머신러닝용 더미변수로 이해하세요)
from sklearn.preprocessing import OneHotEncoder

### 2. 데이터 불러오기 

In [3]:
origin = load_data('nursing_grades')
origin

📚 어느 간호학과 대학원에 지원한 학생들에 대한 합격/불합격 여부를 조사한 가상의 데이터(메타데이터 없음)


,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부
0,NRS0001,장은우,남,380,3.610,3,불합격
1,NRS0002,최지호,남,660,3.670,3,합격
2,NRS0003,김하준,남,800,4.000,1,합격
3,NRS0004,임아윤,여,640,3.190,4,합격
4,NRS0005,강하준,남,520,2.930,4,불합격
...,...,...,...,...,...,...,...
395,NRS0396,박지유,여,620,4.000,2,불합격
396,NRS0397,조하은,여,560,3.040,3,불합격
397,NRS0398,박하윤,여,460,2.630,2,불합격
398,NRS0399,이지우,여,700,3.650,2,불합격


### 3. 범주형 타입 변환 

- 타입변환이 필수는 아니지만, 전체 분석 과정에서는 가급적 수행하는 것이 좋다. 
    - 기술 통계량 산출에 이점이 있기 때문 

In [4]:
타입변환_df = origin.astype({'성별':'category', '합격여부':'category'})
타입변환_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   접수코드    400 non-null    str     
 1   이름      400 non-null    str     
 2   성별      400 non-null    category
 3   필기점수    400 non-null    int64   
 4   학부성적    400 non-null    float64 
 5   병원경력    400 non-null    int64   
 6   합격여부    400 non-null    category
dtypes: category(2), float64(1), int64(2), str(2)
memory usage: 22.9 KB


## #02. 데이터 라벨링 

### 1. 성별에 대한 값 종류 확인 
- 명목형 값의 오름차순으로 정렬 

In [6]:
vcounts = 타입변환_df['성별'].value_counts()
values = vcounts.sort_index()
values

성별
남    194
여    206
Name: count, dtype: int64

### 2. 성별에 대한 라벨링용 딕셔너리 생성 
- values의 index를 라벨링에 적합한 형태로 변환 
    - {"기본값": 변환될 값, "기존값": 변환될 값,... } 형식의 딕셔너리 구조 
- values의 인덱스는 '성별'의 고유값 

In [8]:
labels = {}

for i, v in enumerate(values.index):
    labels[v] = i

labels 

{'남': 0, '여': 1}

### 3. 성별 변수에 라벨링 적용 
- 준비된 딕셔너리를 컬럼 객체(Series)의 map() 메서드에 파라미터로 전달한다. 

In [9]:
라벨링_df = 타입변환_df.copy()
라벨링_df['성별'] = 라벨링_df['성별'].map(labels)
라벨링_df['성별'].value_counts().sort_index()

성별
0    194
1    206
Name: count, dtype: int64

### 4. '합격여부'변수에 대한 일괄 처리

In [10]:
vcounts =라벨링_df['합격여부'].value_counts()
values = vcounts.sort_index()

labels = {}

for i, v in enumerate(values.index):
    labels[v] = i

라벨링_df['합격여부'] = 라벨링_df['합격여부'].map(labels)
라벨링_df['합격여부'].value_counts().sort_index()

합격여부
0     23
1    250
2    127
Name: count, dtype: int64

### 5. 라벨링이 적용된 데이터 프레임 확인
- 각 변수에 대해 라벨링 된 값이 어떤 의미인지 따로 정리해 두어야 한다. 

In [11]:
라벨링_df

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부
0,NRS0001,장은우,0,380,3.610,3,1
1,NRS0002,최지호,0,660,3.670,3,2
2,NRS0003,김하준,0,800,4.000,1,2
3,NRS0004,임아윤,1,640,3.190,4,2
4,NRS0005,강하준,0,520,2.930,4,1
...,...,...,...,...,...,...,...
395,NRS0396,박지유,1,620,4.000,2,1
396,NRS0397,조하은,1,560,3.040,3,1
397,NRS0398,박하윤,1,460,2.630,2,1
398,NRS0399,이지우,1,700,3.650,2,1


## #03. Pandas를 사용하는 더미변수 생성 방법 

### 1. 하나의 컬럼에 대해 모든 값을 더미변수로 변환(값의 수에 따라 N개 생성)

In [12]:
더미변수_df_1 = get_dummies(라벨링_df, columns=['성별'], dtype='int')
더미변수_df_1.head()

,접수코드,이름,필기점수,학부성적,병원경력,합격여부,성별_0,성별_1
0,NRS0001,장은우,380,3.610,3,1,1,0
1,NRS0002,최지호,660,3.670,3,2,1,0
2,NRS0003,김하준,800,4.000,1,2,1,0
3,NRS0004,임아윤,640,3.190,4,2,0,1
4,NRS0005,강하준,520,2.930,4,1,1,0


### 2. 두 개 이상의 명목형 변수 처리 
- 컬럼 이름을 리스트로 묶어서 지정한다. 

In [13]:
더미변수_df_2 = get_dummies(라벨링_df, columns=['성별', '합격여부'], dtype='int')
더미변수_df_2.head()

,접수코드,이름,필기점수,학부성적,병원경력,성별_0,성별_1,합격여부_0,합격여부_1,합격여부_2
0,NRS0001,장은우,380,3.610,3,1,0,0,1,0
1,NRS0002,최지호,660,3.670,3,1,0,0,0,1
2,NRS0003,김하준,800,4.000,1,1,0,0,0,1
3,NRS0004,임아윤,640,3.190,4,0,1,0,0,1
4,NRS0005,강하준,520,2.930,4,1,0,0,1,0


### N-1개의 더미변수 생성 
- drop_first = True 파라미터를 설정한다. (기본값=False) 
    - 성별과 같이 값의 종류가 두개로 구성된 이진 변수는 더미변수 처리를 생략해도 무관하다.

In [14]:
더미변수_df_3 = get_dummies(라벨링_df, columns=['성별', '합격여부'], dtype='int', drop_first=True)
더미변수_df_3.head()

,접수코드,이름,필기점수,학부성적,병원경력,성별_1,합격여부_1,합격여부_2
0,NRS0001,장은우,380,3.610,3,0,1,0
1,NRS0002,최지호,660,3.670,3,0,0,1
2,NRS0003,김하준,800,4.000,1,0,0,1
3,NRS0004,임아윤,640,3.190,4,1,0,1
4,NRS0005,강하준,520,2.930,4,0,1,0


## #04. Scikit-Learn 을 사용한 단일 컬럼의 인코딩 

### 1. 인코딩 대상 변수 추출 

In [15]:
X = 타입변환_df[['성별']]
X.head()

,성별
0,남
1,남
2,남
3,여
4,남


### 2. OneHotEncoding 처리

In [16]:
encoder = OneHotEncoder(sparse_output=False, drop=None, dtype=int)
result = encoder.fit_transform(X)
#출력 량이 많으므로 10개만 출력함 
result[:10]

array([[1, 0],
       [1, 0],
       [1, 0],
       [0, 1],
       [1, 0],
       [0, 1],
       [0, 1],
       [0, 1],
       [0, 1],
       [1, 0]])

## #3. 인코딩 결과를 데이터 프레임으로 생성 

In [17]:
# 컬럼명 생성 
new_cols = encoder.get_feature_names_out(['성별'])
new_cols

array(['성별_남', '성별_여'], dtype=object)

In [18]:
# DataFrame 병합 
one_hot_df = DataFrame(result, columns=new_cols, index=타입변환_df.index)
one_hot_df

,성별_남,성별_여
0,1,0
1,1,0
2,1,0
3,0,1
4,1,0
...,...,...
395,0,1
396,0,1
397,0,1
398,0,1


### 4. 원본 데이터 프레임과 병합 
- 원본 컬럼(성별)이 그대로 남아있기 때문에 drop()함수를 사용해서 삭제할 필요가 있다. 

In [19]:
인코딩_df_1 = merge(타입변환_df, one_hot_df, left_index=True, right_index=True)
인코딩_df_1.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,성별_남,성별_여
0,NRS0001,장은우,남,380,3.610,3,불합격,1,0
1,NRS0002,최지호,남,660,3.670,3,합격,1,0
2,NRS0003,김하준,남,800,4.000,1,합격,1,0
3,NRS0004,임아윤,여,640,3.190,4,합격,0,1
4,NRS0005,강하준,남,520,2.930,4,불합격,1,0


### 5. 불필요한 컬럼 제거 

In [20]:
인코딩_df_1_최종 = 인코딩_df_1.drop(columns=['성별'])
인코딩_df_1_최종.head()

,접수코드,이름,필기점수,학부성적,병원경력,합격여부,성별_남,성별_여
0,NRS0001,장은우,380,3.610,3,불합격,1,0
1,NRS0002,최지호,660,3.670,3,합격,1,0
2,NRS0003,김하준,800,4.000,1,합격,1,0
3,NRS0004,임아윤,640,3.190,4,합격,0,1
4,NRS0005,강하준,520,2.930,4,불합격,1,0


## #05. Scikit-Learn을 사용한 여러 컬럼의 인코딩 

### 1. 여러 컬럼에 대한 일괄 처리 

In [21]:
X = 타입변환_df[['성별', '합격여부']]

encoder = OneHotEncoder(sparse_output=False, drop=None, dtype='int')
result = encoder.fit_transform(X)

result

array([[1, 0, 0, 1, 0],
       [1, 0, 0, 0, 1],
       [1, 0, 0, 0, 1],
       ...,
       [0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0],
       [0, 1, 0, 1, 0]], shape=(400, 5))

### 2. 컬럼명 생성

In [22]:
new_cols = encoder.get_feature_names_out(['성별', '합격여부'])
new_cols

array(['성별_남', '성별_여', '합격여부_보류', '합격여부_불합격', '합격여부_합격'], dtype=object)

### 3. 데이터 프레임으로 병합 

In [23]:
one_hot_df = DataFrame(result, columns=new_cols, index=타입변환_df.index)
one_hot_df.head()

,성별_남,성별_여,합격여부_보류,합격여부_불합격,합격여부_합격
0,1,0,0,1,0
1,1,0,0,0,1
2,1,0,0,0,1
3,0,1,0,0,1
4,1,0,0,1,0


### 4. 원본 데이터 프레임과 병합 

In [24]:
인코딩_df_2 = merge(타입변환_df, one_hot_df, left_index=True, right_index=True)
인코딩_df_2.head()

,접수코드,이름,성별,필기점수,학부성적,병원경력,합격여부,성별_남,성별_여,합격여부_보류,합격여부_불합격,합격여부_합격
0,NRS0001,장은우,남,380,3.610,3,불합격,1,0,0,1,0
1,NRS0002,최지호,남,660,3.670,3,합격,1,0,0,0,1
2,NRS0003,김하준,남,800,4.000,1,합격,1,0,0,0,1
3,NRS0004,임아윤,여,640,3.190,4,합격,0,1,0,0,1
4,NRS0005,강하준,남,520,2.930,4,불합격,1,0,0,1,0


### 5. 불필요한 컬럼 제거 

In [25]:
인코딩_df_2.drop(columns=['성별', '합격여부'], inplace=True) 
인코딩_df_2.head()

,접수코드,이름,필기점수,학부성적,병원경력,성별_남,성별_여,합격여부_보류,합격여부_불합격,합격여부_합격
0,NRS0001,장은우,380,3.610,3,1,0,0,1,0
1,NRS0002,최지호,660,3.670,3,1,0,0,0,1
2,NRS0003,김하준,800,4.000,1,1,0,0,0,1
3,NRS0004,임아윤,640,3.190,4,0,1,0,0,1
4,NRS0005,강하준,520,2.930,4,1,0,0,1,0


## #01. 연습문제 - 온라인 스토어 고객 행동 분석 

In [ ]:
customers_df = load_data('online_store_customers')
customers_df

📚 한 온라인 스토어의 마케팅 팀이 연말 특별 프로모션을 기획에 필요한 고객의 기본 인구 통계 정보(메타데이터 없음)


,name,gender,age
user_id,,,
1,Alice,F,25
2,Bob,M,30
3,Charlie,M,35
4,David,M,42
5,Eve,F,28
6,Frank,M,21
7,Grace,F,33
8,Henry,M,45
9,Ivy,F,29


In [27]:
purchases_df = load_data('online_store_purchases')
purchases_df

📚 한 온라인 스토어의 마케팅 팀이 연말 특별 프로모션을 기획에 필요한 고객의 구매 관련 정보(메타데이터 없음)


,user_id,product,size,color,price
purchase_id,,,,,
101,1,T-shirt,M,White,15000
102,2,Pants,L,Black,35000
103,1,Skirt,S,Red,25000
104,3,T-shirt,L,Blue,17000
105,5,Jacket,M,Black,55000
106,4,Pants,M,Khaki,33000
107,6,T-shirt,S,White,16000
108,7,Jacket,NaN,Red,58000
109,8,Pants,L,Black,38000


In [29]:
병합_df = concat(customers_df, purchases_df)
병합_df

NameError: name 'concat' is not defined